In [ ]:
from pathlib import Path
import random

import albumentations as A
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader, Dataset

import sys
sys.path.insert(0, str(Path("../src").resolve()))
from models import UNet

In [ ]:
SEED = 42
IMAGE_SIZE = 256
BATCH_SIZE = 16
EPOCHS = 10
BILINEAR = False

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_ROOT = Path("../brisc2025")
SPLIT_ROOT = Path("../data")
CHECKPOINT_ROOT = Path("../checkpoints")
CHECKPOINT_ROOT.mkdir(exist_ok=True)

device = (
    torch.device("cuda") if torch.cuda.is_available()
    else torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cpu")
)

print("Device:", device)

In [ ]:
train_df = pd.read_csv(SPLIT_ROOT / "segmentation_train.csv")
val_df = pd.read_csv(SPLIT_ROOT / "segmentation_val.csv")

print("Full training samples:", len(train_df))
print("Full validation samples:", len(val_df))

In [ ]:
train_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=10, border_mode=cv2.BORDER_CONSTANT, p=0.5),
    A.Normalize(mean=(0.5,), std=(0.5,)),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.Normalize(mean=(0.5,), std=(0.5,)),
    ToTensorV2(),
])

In [ ]:
class SegmentationDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image_path = DATA_ROOT / row["relative_path"]
        mask_path = DATA_ROOT / row["mask_relative_path"]

        image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

        if image is None or mask is None:
            raise FileNotFoundError(f"Could not load {image_path} or {mask_path}")

        mask = (mask > 0).astype(np.float32)
        transformed = self.transform(image=image, mask=mask)
        
        image = transformed["image"].float()
        mask = transformed["mask"].float()

        if mask.ndim == 2:
            mask = mask.unsqueeze(0)

        return image, mask

In [ ]:

train_dataset = SegmentationDataset(train_df, train_transform)
val_dataset = SegmentationDataset(val_df, val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)
images, masks = next(iter(train_loader))

## U-Net


In [ ]:
model = UNet(n_channels=1, n_classes=1, bilinear=BILINEAR).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## metric

In [ ]:
def dice_coefficient(logits, targets, threshold=0.5, smooth=1e-6):
    predictions = (torch.sigmoid(logits) >= threshold).float()
    dims = (1, 2, 3)
    intersection = (predictions * targets).sum(dim=dims)
    denominator = predictions.sum(dim=dims) + targets.sum(dim=dims)
    return ((2 * intersection + smooth) / (denominator + smooth)).mean()


def segmentation_loss(logits, targets, smooth=1e-6):
    bce = F.binary_cross_entropy_with_logits(logits, targets)
    probabilities = torch.sigmoid(logits)
    dims = (1, 2, 3)
    intersection = (probabilities * targets).sum(dim=dims)
    denominator = probabilities.sum(dim=dims) + targets.sum(dim=dims)
    dice_loss = 1 - ((2 * intersection + smooth) / (denominator + smooth)).mean()
    return 0.5 * bce + 0.5 * dice_loss

# test 
with torch.no_grad():
    test_logits = model(images.to(device))

print("Model output:", test_logits.shape)
print("Initial loss:", segmentation_loss(test_logits, masks.to(device)).item())
assert test_logits.shape == masks.shape

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)


def run_epoch(loader, training):
    model.train(training)
    total_loss = 0.0
    total_dice = 0.0

    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            logits = model(images)
            loss = segmentation_loss(logits, masks)

            if training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * images.size(0)
        total_dice += dice_coefficient(logits, masks).item() * images.size(0)

    size = len(loader.dataset)
    return total_loss / size, total_dice / size

In [ ]:

best_dice = -1.0

for epoch in range(EPOCHS):
    train_loss, train_dice = run_epoch(train_loader, training=True)
    eval_loss, eval_dice = run_epoch(val_loader, training=False)

    if eval_dice > best_dice:
        best_dice = eval_dice


    print(
        f"Epoch {epoch + 1:02d}: "
        f"train loss={train_loss:.4f}, train Dice={train_dice:.4f}, "
        f"eval loss={eval_loss:.4f}, eval Dice={eval_dice:.4f}"
    )

print(best_dice)

In [ ]:

model.eval()

images, masks = next(iter(val_loader))
with torch.no_grad():
    probabilities = torch.sigmoid(model(images.to(device))).cpu()

predictions = (probabilities >= 0.5).float()
count = min(4, len(images))
fig, axes = plt.subplots(count, 3, figsize=(10, 3 * count))

if count == 1:
    axes = axes[None, :]

for index in range(count):
    image = images[index, 0].numpy() * 0.5 + 0.5
    axes[index, 0].imshow(image, cmap="gray")
    axes[index, 0].set_title("MRI")
    axes[index, 1].imshow(masks[index, 0], cmap="gray")
    axes[index, 1].set_title("Ground truth")
    axes[index, 2].imshow(predictions[index, 0], cmap="gray")
    axes[index, 2].set_title("Prediction")

for axis in axes.flat:
    axis.axis("off")

plt.tight_layout()
plt.show()